# RT-DETR Tennis Racquet Fine-tuning

**Before running:** Runtime → Change runtime type → **T4 GPU**

### Prerequisites

Upload these three files to Google Drive inside a folder — e.g. `My Drive/rtdetr-finetune/`:
- `dataset.py`
- `train.py`
- `ap-tennis.v1i.coco.zip`

Then set `DRIVE` in the Config cell and run all cells top to bottom.

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — change runtime type to T4 GPU before continuing.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Edit these settings before running ────────────────────────────────────────
DRIVE = '/content/drive/MyDrive/rtdetr-finetune'

os.environ['OUTPUT_DIR']  = f'{DRIVE}/rtdetr-tennis'
os.environ['BATCH_SIZE']  = '16'   # T4 has 16 GB VRAM; drop to 8 if OOM
os.environ['MAX_EPOCHS']  = '60'
os.environ['NUM_WORKERS'] = '4'

print(f"OUTPUT_DIR  : {os.environ['OUTPUT_DIR']}")
print(f"BATCH_SIZE  : {os.environ['BATCH_SIZE']}")
print(f"MAX_EPOCHS  : {os.environ['MAX_EPOCHS']}")
print(f"NUM_WORKERS : {os.environ['NUM_WORKERS']}")

In [ ]:
!pip install -q 'transformers>=4.40' 'torchmetrics>=1.9.0' 'accelerate>=1.13.0' scipy pillow faster-coco-eval

In [ ]:
import shutil, os, subprocess

shutil.copy(f'{DRIVE}/dataset.py', '/content/dataset.py')
shutil.copy(f'{DRIVE}/train.py',   '/content/train.py')
print('Source files copied.')

if not os.path.exists('/content/data/train'):
    print('Extracting dataset (this takes ~1 min)...')
    subprocess.run(
        ['unzip', '-q', f'{DRIVE}/ap-tennis.v1i.coco.zip', '-d', '/content/data'],
        check=True,
    )
    print('Done.')
else:
    print('Dataset already extracted, skipping.')

for split in sorted(os.listdir('/content/data')):
    split_path = f'/content/data/{split}'
    if os.path.isdir(split_path):
        n = len([f for f in os.listdir(split_path) if f.endswith('.jpg')])
        print(f'  {split}/  — {n} images')

In [ ]:
%cd /content
!python train.py

## Resuming After a Session Disconnect

Free Colab sessions time out after ~12 hours. Checkpoints are saved directly to Drive after every epoch, so no work is lost.

To resume: re-run all cells above (GPU check → Config → pip install → copy files), then run the cell below.

In [ ]:
import glob, os

output_dir  = os.environ.get('OUTPUT_DIR', f'{DRIVE}/rtdetr-tennis')
checkpoints = sorted(glob.glob(f'{output_dir}/checkpoint-*'), key=os.path.getmtime)

if checkpoints:
    latest = checkpoints[-1]
    print(f'Resuming from: {latest}')
    # Remove stale GradScaler state that can break fp16 resume across Accelerate versions.
    scaler_path = os.path.join(latest, 'scaler.pt')
    if os.path.exists(scaler_path):
        os.remove(scaler_path)
        print('  Removed stale scaler.pt')
else:
    print('No checkpoint found — train.py will start from scratch.')

%cd /content
!python train.py